# Structure Investigation

In [1]:
%load_ext autoreload
%autoreload 2

import t2fpharm_study
import pandas as pd

In [2]:
manager = t2fpharm_study.manager()

In [3]:
p=manager.complex("1aq1")

In [7]:
p.composition.chain_ids().to_numpy(dtype=str)

array(['A'], dtype='<U1')

In [17]:
import pandas as pd
df=pd.DataFrame([{"a":"A","x":False},{"a":"A","x":False},{"a":"B","x":False}])
df[df["x"]]["a"].unique()

array([], dtype=object)

In [19]:
p.composition.atoms

,chain_id,res_name,res_seq,i_code,res_poly,res_std,serial,name,alt_loc,occupancy,temp_factor,element,charge,element_index
serial,,,,,,,,,,,,,,
1,A,MET,1,,True,True,1,N,,1,0,N,<NA>,6
2,A,MET,1,,True,True,2,H,,1,0,H,<NA>,0
3,A,MET,1,,True,True,3,H2,,1,0,H,<NA>,0
4,A,MET,1,,True,True,4,H3,,1,0,H,<NA>,0
5,A,MET,1,,True,True,5,CA,,1,0,C,<NA>,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5281,A,HOH,423,,False,False,5281,H1,,1,0,H,<NA>,0
5282,A,HOH,423,,False,False,5282,H2,,1,0,H,<NA>,0
5283,A,HOH,424,,False,False,5283,O,,1,0,O,<NA>,7


In [20]:
p.composition.sequence()

array(['MET', 'GLU', 'ASN', 'PHE', 'GLN', 'LYS', 'VAL', 'GLU', 'LYS',
       'ILE', 'GLY', 'GLU', 'GLY', 'THR', 'TYR', 'GLY', 'VAL', 'VAL',
       'TYR', 'LYS', 'ALA', 'ARG', 'ASN', 'LYS', 'LEU', 'THR', 'GLY',
       'GLU', 'VAL', 'VAL', 'ALA', 'LEU', 'LYS', 'LYS', 'ILE', 'ARG',
       'LEU', 'ASP', 'THR', 'GLU', 'THR', 'GLU', 'GLY', 'VAL', 'PRO',
       'SER', 'THR', 'ALA', 'ILE', 'ARG', 'GLU', 'ILE', 'SER', 'LEU',
       'LEU', 'LYS', 'GLU', 'LEU', 'ASN', 'HIS', 'PRO', 'ASN', 'ILE',
       'VAL', 'LYS', 'LEU', 'LEU', 'ASP', 'VAL', 'ILE', 'HIS', 'THR',
       'GLU', 'ASN', 'LYS', 'LEU', 'TYR', 'LEU', 'VAL', 'PHE', 'GLU',
       'PHE', 'LEU', 'HIS', 'GLN', 'ASP', 'LEU', 'LYS', 'LYS', 'PHE',
       'MET', 'ASP', 'ALA', 'SER', 'ALA', 'LEU', 'THR', 'GLY', 'ILE',
       'PRO', 'LEU', 'PRO', 'LEU', 'ILE', 'LYS', 'SER', 'TYR', 'LEU',
       'PHE', 'GLN', 'LEU', 'LEU', 'GLN', 'GLY', 'LEU', 'ALA', 'PHE',
       'CYS', 'HIS', 'SER', 'HIS', 'ARG', 'VAL', 'LEU', 'HIS', 'ARG',
       'ASP', 'LEU',

In [ ]:
manager.load()

In [ ]:
def atoms(pdb_id):
    return manager.complex(pdb_id.upper()).composition.atoms

def chains(pdb_id):
    return atoms(pdb_id)["chain_id"].unique()

def hets(pdb_id):
    _atoms = atoms(pdb_id)
    hets = _atoms[~_atoms["res_poly"]].copy()
    hets['res_seq'] = hets['res_seq'].astype(int)
    hets_unique = hets.drop_duplicates(subset=['res_name','chain_id','res_seq'])
    result = [
        (
            res_name,
            [
                (chain_id, list(chunk['res_seq']))        # for each chain in this residue
                for chain_id, chunk in group.groupby('chain_id')
            ]
        )
        for res_name, group in hets_unique.groupby('res_name')
    ]
    return sorted(result, key=lambda x: x[0] == "HOH")

def identical(pdb_id1, pdb_id2, chain_id1 = None, chain_id2 = None):
    atoms1 = atoms(pdb_id1)
    atoms2 = atoms(pdb_id2)
    if not chain_id1:
        chains1 = chains(pdb_id1)
        if chains1.size != 1:
            raise ValueError(f"{pdb_id1} has more than one chain: {chains1}")
        chain_id1 = chains1[0]
    if not chain_id2:
        chains2 = chains(pdb_id2)
        if chains2.size != 1:
            raise ValueError(f"{pdb_id2} has more than one chain: {chains2}")
        chain_id2 = chains2[0]
    chain_atoms1 = atoms1[
        (atoms1["chain_id"] == chain_id1) & atoms1["res_poly"]
    ].reset_index(drop=True)
    chain_atoms2 = atoms2[
        (atoms2["chain_id"] == chain_id2) & atoms2["res_poly"]
    ].reset_index(drop=True)
    # colums = ["res_name", "name", "element"]
    # mismatch_mask = (chain_atoms1[colums] != chain_atoms2[colums]).any(axis=1)
    # if mismatch_mask.any():
    #     first_bad_idx = mismatch_mask.idxmax()
    #     return (
    #         int(chain_atoms1.iloc[first_bad_idx]["serial"]),
    #         int(chain_atoms2.iloc[first_bad_idx]["serial"]),
    #     )
    # return True
    return compare_dataframes(chain_atoms1, chain_atoms2)

def compare_dataframes(
    df1: pd.DataFrame,
    df2: pd.DataFrame
) -> bool | tuple:
    """
    Compare two DataFrames by 'res_name', then within each contiguous group of the same 'res_name',
    check that the sets of ('name','element') match. Return the first mismatch serials or True.

    Parameters
    ----------
    df1 : pd.DataFrame
        First DataFrame containing columns 'res_name', 'name', 'element', 'serial'.
    df2 : pd.DataFrame
        Second DataFrame with the same columns.

    Returns
    -------
    True
        If all checks pass.
    Tuple[Any, Any]
        A tuple (serial1, serial2) of the values at the first mismatch.
    """
    # Step 1: row-by-row comparison of 'res_name'
    df1 = df1.reset_index(drop=True)
    df2 = df2.reset_index(drop=True)
    n1, n2 = len(df1), len(df2)
    min_len = min(n1, n2)
    for i in range(min_len):
        if df1['res_name'].iat[i] != df2['res_name'].iat[i]:
            return df1['serial'].iat[i], df2['serial'].iat[i]
    if n1 != n2:
        # lengths differ -> mismatch at first extra row
        idx = min_len
        s1 = df1['serial'].iat[idx] if n1 > idx else None
        s2 = df2['serial'].iat[idx] if n2 > idx else None
        return s1, s2

    # Step 2: group contiguous identical 'res_name'
    group_ids = (df1['res_name'] != df1['res_name'].shift()).cumsum()
    for gid in group_ids.unique():
        idxs = group_ids[group_ids == gid].index
        # sets of (name,element)
        s1 = set(zip(df1.loc[idxs, 'name'], df1.loc[idxs, 'element']))
        s2 = set(zip(df2.loc[idxs, 'name'], df2.loc[idxs, 'element']))
        if s1 != s2:
            # find first mismatch index
            mismatches = [i for i in idxs
                          if (df1.at[i, 'name'], df1.at[i, 'element']) not in s2
                          or (df2.at[i, 'name'], df2.at[i, 'element']) not in s1]
            if mismatches:
                first_i = min(mismatches)
                return df1.at[first_i, 'serial'], df2.at[first_i, 'serial']

    # All checks passed
    return True

In [ ]:
for group_id, group in manager.dataset.groupby("group_id", sort=False):
    print("="*100)
    print(group_id)
    print("="*100)
    ref_row = group[group["is_ref"]].iloc[0]
    ref_pdb_id = ref_row["pdb_id"]
    for _, row in group.iterrows():
        pdb_id = row["pdb_id"]
        # print("-"*50)
        # print(pdb_id)
        # print("-"*50)
        # print("Chains:", ", ".join(chains(pdb_id).tolist()))
        # print("Hets:")
        # for het in hets(pdb_id):
        #     print(f"  {het[0]} ({len(het[1])})")
        #     for chain_id, res_seqs in het[1]:
        #         print(f"    {chain_id} ({len(res_seqs)})")
        #         for res_seq in res_seqs:
        #             print(f"      {res_seq}")        
        if row["is_ref"]:
            continue
        print("Identity:", identical(ref_pdb_id, pdb_id))

In [ ]:
nv=manager.complex("1vru").display()
nv

In [ ]:
nv, _ = manager.ligand_pharmacophore("1bqm").extra["plip"].display()
nv.add_trajectory(manager.complex("1bqm"))
nv.display(gui=True)

In [ ]:
comp = manager.complex("1fpc")
atoms = comp.composition.atoms

In [ ]:
a=atoms[atoms["chain_id"] == "A"]
c=atoms[atoms["chain_id"] == "B"]

In [ ]:
atoms[atoms["res_name"]=="0ZI"]

In [ ]:
compare_dataframes(a[a["res_poly"]], c[c["res_poly"]])

In [ ]:
a[a["res_poly"]].shape, c[c["res_poly"]].shape

In [ ]:
a[a["res_poly"] & (a["res_seq"] == 127)]

In [ ]:
c[c["res_poly"] & (c["res_seq"] == 127)]

## CDK2

### 1AQ1

In [ ]:
chains("1aq1")[0]

In [ ]:
hets("1aq1")

### 1DI8

In [ ]:
chains("1di8")

In [ ]:
hets("1di8")

In [ ]:
identical("1aq1", "1di8")